In [1]:
import time
import csv
# write the tabular data into a csv file
from pathlib import Path
# handle file system paths

In [2]:
from selenium import webdriver
# control the browser
from selenium.webdriver.chrome.options import Options
# configure the chrome behavior
from selenium.webdriver.common.by import By
# locate elements on a web page
from selenium.webdriver.support.ui import WebDriverWait
# to wait for conditions
from selenium.webdriver.support import expected_conditions as EC
# define tha wait logic

In [3]:
URL="https://airquality.cpcb.gov.in/AQI_India/"
OUT_FILE="res.csv"
PROFILE_DIR=str(Path.cwd()/"chrome_profile_cpcb")
# defines a persistent profile folder a persistent profile allows cookies to be reused

3. define the driver method


In [10]:
def start_deiver():
    options=Options()
    # chrome launch settings are stored
    options.add_argument('--window-size=1920,1080')
    # chrome viewport is forced to 1920 and 1080
    options.add_argument(f'--user-data-dir={PROFILE_DIR}')
    # adds a user data directory it is used to store cookiies, cache local storeage, session details
    options.add_argument('--disable-gpu')
    # used to didavle GUP accelearation to reduce rendering issues
    options.add_argument('--log-level=3')
    # reduce the log output only higher severity cases are logged
    driver=webdriver.Chrome(options=options)
    # a chrome webdriver is created ti launches a window with the options 
    # the drive object is returned to the caller for further automation
# 4. define the close blocking_modal function
def close_blocking_modal(driver):
    # parameter passed is the selenium webdriver 
    time.sleep(1)
    close_xpath=[
        "//div[@id='myModel']//button[contains(@class,'close')]",
        "//div[@id='myModel']//button[contains(.,'Close')]",
        "//div[@id='myModel']//button[contains(.,'OK')]",
        "//div[@class='modal-footer']//button[contains(.,'Agree')]",
        "//div[@id='myModel']//a[contains(.,'Close')]"
    ]
    # it store multiple Xpath patterns for close contorels this targets the x icon
    for xp in close_xpath:
        try:
            btn = WebDriverWait(driver, 3).until(
                EC.element_to_be_clickable((By.XPATH, xp))
            )
            # webdriver wait is created for upto 3 seconds the condition ewquires the element to be located by the current XPATH
            driver.execute_script("arguments[0].click();", btn)
            # javascript is executed in this page 
            # arguments[0] is replaced by the btn
            # click is triggered on the element through javascript 
            time.sleep(1)
            return
        except:
            pass
        driver.execute_script("""
                const m = document.querySelector('#myModel');
                if(m): m.remove();
                const b = document.querySelector('.modal-backdrop');
                if(b): b.remove();
                document.body.classList.remove('modal-open');
                """)
        # searches for a modal element with id myMoodel if it exists it is removed from the DOM
        #  the dark overlay behind the model id searched if it exists it is removed
        # the model - open class is removed from the body this restores the scroll and click behavior
        time.sleep(1)
#  5. define a function to extract the table 
def extract_table(driver):
    # parameter is the selenium webdriver
    tables=driver.find_elements(By.TAG_NAME,"table")
    # all table elements in the current page are collected find_elements returns a list 
    # by_elements returns a list
    best = None
    # holds the best table WebElement
    best_rows = 0
    for t in tables:
        rows=t.find_elements(By.CLASS_NAME,'tr')
        # for the current table all row elements are selected
        if len(rows)>best_rows:
            best_rows=len(rows)
            best=t
        if not best or best_rows<2:
            # any table should be have a minimum of 2 rows
            # one row for header, one row for actual data
            raise RuntimeError('no populated table found')
        rows=best.find_elements(By.CLASS_NAME,'tr')
        header=[c.text.strip() for c in rows[0].find_elements(By.CSS_SELECTOR,'th,td')]
        data=[]
        for r in rows[1:]:
            cells=[c.text.strip().replace('\n',' ') for c in r.find_elements(By.CSS_SELECTOR,'th,td')]
            if any(cells):
                # if any cell has content 
                data.append(cells)
        return header,data
    
# 5.5 check whether tha table has data
def has_populated_table(driver):
    for t in driver.find_elements(By.TAG_NAME,'table'):
        rows=t.find_elements(By.CSS_SELECTOR,'tr')
        if len(rows)>=2 and t.text.strip():
            return True
        return False    
# 6. staart the browser session
driver=start_deiver()
try:
    driver.get(URL)
    print('wait for the link to be clicked')
    close_blocking_modal(driver)
    WebDriverWait(driver,300).until(lambda d:d.execute_script('return document.readyState')=='complete')
    # lambda is executed repeatedly to check the ready state of the document
    # wait block ends and execution resumes after the page loads
    repo_link=WebDriverWait(driver,300).until(EC.element_to_be_clickable((By.XPATH,'//a[contains(.,"/AQI DATA Repository")]')))
    # all anchor tagrs containing the specfic text is targeted the link has to be clickable
    close_blocking_modal(driver)
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});",repo_link)
    time.sleep(0.5)
    driver.execute_script("arguments[0].click();",repo_link)
    WebDriverWait(driver,10).until(lambda d:len(d.window_handles)>=1)
    if len(driver.window_handles)>1:
        driver.switch_to.window(driver.window_handles[-1])
        # switch to the last opened tab
    print('page loaded')
    print(f'URL:{driver.current_url}')
    print(f'title:{driver.title}')
    WebDriverWait(driver,180,poll_frequency=1).until(has_populated_table)
    # poll_frequency 1 means the check runs once per second
    # has populated table is called repeatedlty until it return a true value or timeout occurs
    header,rows = extract_table(driver)
    with open(OUT_FILE,'w',newline='',encoding='utf-8') as f:
        writer=csv.writer(f)
        writer.writerow(header)
        writer.writerows(rows)
        print(f'saved {len(rows)} to {OUT_FILE}')
except Exception as e:
    Path('debug_repo_page.html').write_text(driver.page_source,encoding='utf-8')
    driver.save_screenshot('debug_repo_page.png')
    print('Extration failed,debug file saved ')
    raise e
finally:
    driver.quit()

AttributeError: 'NoneType' object has no attribute 'quit'

In [4]:
# ------------------------------------------
# 1. START DRIVER
# ------------------------------------------
def start_driver():
    options = Options()

    options.add_argument('--window-size=1920,1080')
    options.add_argument(f'--user-data-dir={PROFILE_DIR}')
    options.add_argument('--disable-gpu')
    options.add_argument('--log-level=3')

    driver = webdriver.Chrome(options=options)
    return driver      # 🔥 THIS WAS MISSING 🔥


# ------------------------------------------
# 2. CLOSE MODAL POPUP
# ------------------------------------------
def close_blocking_modal(driver):
    time.sleep(1)

    close_xpath = [
        "//div[@id='myModel']//button[contains(@class,'close')]",
        "//div[@id='myModel']//button[contains(.,'Close')]",
        "//div[@id='myModel']//button[contains(.,'OK')]",
        "//div[@class='modal-footer']//button[contains(.,'Agree')]",
        "//div[@id='myModel']//a[contains(.,'Close')]"
    ]

    for xp in close_xpath:
        try:
            btn = WebDriverWait(driver, 3).until(
                EC.element_to_be_clickable((By.XPATH, xp))
            )
            driver.execute_script("arguments[0].click();", btn)
            time.sleep(1)
            return
        except:
            pass

    # force remove modal if button not found
    driver.execute_script("""
        const m = document.querySelector('#myModel');
        if(m) m.remove();
        const b = document.querySelector('.modal-backdrop');
        if(b) b.remove();
        document.body.classList.remove('modal-open');
    """)


# ------------------------------------------
# 3. CHECK IF TABLE HAS DATA
# ------------------------------------------
def has_populated_table(driver):
    for t in driver.find_elements(By.TAG_NAME, 'table'):
        rows = t.find_elements(By.CSS_SELECTOR, 'tr')
        if len(rows) >= 2 and t.text.strip():
            return True
    return False


# ------------------------------------------
# 4. EXTRACT TABLE
# ------------------------------------------
def extract_table(driver):
    tables = driver.find_elements(By.TAG_NAME, "table")

    best = None
    best_rows = 0

    for t in tables:
        rows = t.find_elements(By.CSS_SELECTOR, 'tr')
        if len(rows) > best_rows:
            best_rows = len(rows)
            best = t

    if not best or best_rows < 2:
        raise RuntimeError("No populated table found")

    rows = best.find_elements(By.CSS_SELECTOR, 'tr')

    header = [c.text.strip() for c in rows[0].find_elements(By.CSS_SELECTOR, 'th,td')]
    data = []

    for r in rows[1:]:
        cells = [c.text.strip().replace('\n', ' ') for c in r.find_elements(By.CSS_SELECTOR, 'th,td')]
        if any(cells):
            data.append(cells)

    return header, data


# ------------------------------------------
# 5. MAIN PROGRAM
# ------------------------------------------
driver = None

try:
    driver = start_driver()
    driver.get(URL)

    print("Waiting for page load...")
    close_blocking_modal(driver)

    WebDriverWait(driver, 300).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )

    repo_link = WebDriverWait(driver, 300).until(
        EC.element_to_be_clickable((By.XPATH, '//a[contains(.,"/AQI DATA Repository")]'))
    )

    close_blocking_modal(driver)

    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", repo_link)
    time.sleep(0.5)
    driver.execute_script("arguments[0].click();", repo_link)

    WebDriverWait(driver, 10).until(lambda d: len(d.window_handles) >= 1)

    if len(driver.window_handles) > 1:
        driver.switch_to.window(driver.window_handles[-1])

    print("Page loaded")
    print("URL:", driver.current_url)
    print("Title:", driver.title)

    WebDriverWait(driver, 180, poll_frequency=1).until(has_populated_table)

    header, rows = extract_table(driver)

    with open(OUT_FILE, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(rows)

    print(f"Saved {len(rows)} rows to {OUT_FILE}")

except Exception as e:
    if driver:
        Path("debug_repo_page.html").write_text(driver.page_source, encoding="utf-8")
        driver.save_screenshot("debug_repo_page.png")
        print("Extraction failed. Debug files saved.")
    raise e

finally:
    if driver:
        driver.quit()


Waiting for page load...


InvalidSessionIdException: Message: invalid session id; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff7c6fa88d5
	0x7ff7c6fa8930
	0x7ff7c6d81465
	0x7ff7c6dc9d83
	0x7ff7c6e01ee2
	0x7ff7c6dfc500
	0x7ff7c6dfb6f0
	0x7ff7c6d4b325
	0x7ff7c72c0620
	0x7ff7c72baf60
	0x7ff7c72d96c6
	0x7ff7c6fc5dd4
	0x7ff7c6fced7c
	0x7ff7c6d49cb8
	0x7ff7c741efe8
	0x7ffc7af6e8d7
	0x7ffc7b9ec3dc
